In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, explode, col, regexp_replace
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS

In [2]:
spark = SparkSession.builder \
    .appName("MIND-ALS") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

In [10]:
schema_behaviors = "impression_id STRING, user_id STRING, time STRING, history STRING, impressions STRING"

behaviors_df = spark.read.csv(
    "behaviors_small.tsv",
    sep="\t",
    schema=schema_behaviors
)

schema_news = "news_id STRING, category STRING, subcategory STRING, title STRING, abstract STRING, url STRING, title_entities STRING, abstract_entities STRING"

news_df = spark.read.csv(
    "news.tsv",
    sep="\t",
    schema=schema_news
)

df = behaviors_df.withColumn(
    "impression",
    explode(split(col("Impressions"), " "))
)

df = df.withColumn("newsId", split(col("impression"), "-")[0]) \
       .withColumn("clicked", split(col("impression"), "-")[1].cast("int"))

In [4]:
df = df.withColumn(
    "userId",
    regexp_replace(col("user_id"), "^U", "")
)

df = df.withColumn(
    "itemId",
    regexp_replace(col("newsId"), "^N", "")
)

df = df.withColumn("userId", col("userId").cast("int")) \
       .withColumn("itemId", col("itemId").cast("int"))

print(df.head(3))
print("\n-----------------------------------------\n")

[Row(impression_id='1', user_id='U87243', time='11/10/2019 11:30:54 AM', history='N8668 N39081 N65259 N79529 N73408 N43615 N29379 N32031 N110232 N101921 N12614 N129591 N105760 N60457 N1229 N64932', impressions='N78206-0 N26368-0 N7578-0 N58592-0 N19858-0 N58258-0 N18478-0 N2591-0 N97778-0 N32954-0 N94157-1 N39404-0 N108809-0 N78699-1 N71090-1 N40282-0 N31174-1 N37924-0 N27822-0', impression='N78206-0', newsId='N78206', clicked=0, userId=87243, itemId=78206), Row(impression_id='1', user_id='U87243', time='11/10/2019 11:30:54 AM', history='N8668 N39081 N65259 N79529 N73408 N43615 N29379 N32031 N110232 N101921 N12614 N129591 N105760 N60457 N1229 N64932', impressions='N78206-0 N26368-0 N7578-0 N58592-0 N19858-0 N58258-0 N18478-0 N2591-0 N97778-0 N32954-0 N94157-1 N39404-0 N108809-0 N78699-1 N71090-1 N40282-0 N31174-1 N37924-0 N27822-0', impression='N26368-0', newsId='N26368', clicked=0, userId=87243, itemId=26368), Row(impression_id='1', user_id='U87243', time='11/10/2019 11:30:54 AM', his

In [ ]:
als_df = df.select(
    col("userId"),
    col("itemId"),
    col("clicked").alias("rating")
)

print(als_df.head(3))


als = ALS(
    userCol="userId",
    itemCol="itemId",
    ratingCol="rating",
    implicitPrefs=True,
    alpha=15,
    rank=20,
    regParam=0.1,
    coldStartStrategy="drop"
)

model = als.fit(als_df)

[Row(userId=87243, itemId=78206, rating=0), Row(userId=87243, itemId=26368, rating=0), Row(userId=87243, itemId=7578, rating=0)]


In [ ]:
user_recs = model.recommendForAllUsers(5)


[Row(userId=950, recommendations=[Row(itemId=107637, rating=0.8671701550483704), Row(itemId=15053, rating=0.8145924210548401), Row(itemId=31879, rating=0.6595297455787659), Row(itemId=47257, rating=0.5897689461708069), Row(itemId=100449, rating=0.5048919916152954)]), Row(userId=1664, recommendations=[Row(itemId=101624, rating=0.41603320837020874), Row(itemId=123683, rating=0.3637506365776062), Row(itemId=25751, rating=0.3553379774093628), Row(itemId=93643, rating=0.28784307837486267), Row(itemId=46716, rating=0.25569167733192444)]), Row(userId=1961, recommendations=[Row(itemId=110627, rating=0.8900083899497986), Row(itemId=100456, rating=0.8864325881004333), Row(itemId=42649, rating=0.7564555406570435), Row(itemId=120147, rating=0.6563361287117004), Row(itemId=76665, rating=0.6402307152748108)]), Row(userId=2531, recommendations=[Row(itemId=123077, rating=0.9590345621109009), Row(itemId=4504, rating=0.8580101132392883), Row(itemId=38378, rating=0.7646823525428772), Row(itemId=51142, ra

In [9]:
print(user_recs.head(5))

[Row(userId=950, recommendations=[Row(itemId=107637, rating=0.8671701550483704), Row(itemId=15053, rating=0.8145924210548401), Row(itemId=31879, rating=0.6595297455787659), Row(itemId=47257, rating=0.5897689461708069), Row(itemId=100449, rating=0.5048919916152954)]), Row(userId=1664, recommendations=[Row(itemId=101624, rating=0.41603320837020874), Row(itemId=123683, rating=0.3637506365776062), Row(itemId=25751, rating=0.3553379774093628), Row(itemId=93643, rating=0.28784307837486267), Row(itemId=46716, rating=0.25569167733192444)]), Row(userId=1961, recommendations=[Row(itemId=110627, rating=0.8900083899497986), Row(itemId=100456, rating=0.8864325881004333), Row(itemId=42649, rating=0.7564555406570435), Row(itemId=120147, rating=0.6563361287117004), Row(itemId=76665, rating=0.6402307152748108)]), Row(userId=2531, recommendations=[Row(itemId=123077, rating=0.9590345621109009), Row(itemId=4504, rating=0.8580101132392883), Row(itemId=38378, rating=0.7646823525428772), Row(itemId=51142, ra

In [ ]:
def get_clicked_news_for_user(user_id: str):
    user_df = behaviors_df.filter(col("user_id") == user_id)

    exploded = (
        user_df
        .select(explode(split(col("impressions"), " ")).alias("impression"))
        .withColumn("news_id", split(col("impression"), "-").getItem(0))
        .withColumn("clicked", split(col("impression"), "-").getItem(1).cast("int"))
        .filter(col("clicked") == 1)
    )

    result = (
        exploded
        .join(news_df, on="news_id", how="inner")
        .select("category", "subcategory", "title")
    )

    return result

In [13]:
results = get_clicked_news_for_user('U950')
results.show()

+------------+------------+--------------------+
|    category| subcategory|               title|
+------------+------------+--------------------+
|foodanddrink|     recipes|What to Cook This...|
|      sports|football_nfl|Russell Wilson vs...|
+------------+------------+--------------------+



In [ ]:
def get_news_info_by_id(news_id: str):
    result = (
        news_df
        .filter(col("news_id") == news_id)
        .select("category", "subcategory", "title")
    )

    return result

In [ ]:
'''get_news_info_by_id("N107637").show()
get_news_info_by_id("N15053").show()
get_news_info_by_id("N31879").show()
get_news_info_by_id("N47257").show()
get_news_info_by_id("N100449").show()'''
print('----------------------------------------------------------')
#N31879
#N3718, N40122, N38304, N87855, N92438, N117260, N47026, N84137, N56196
#452	U244721	11/11/2019 11:57:05 AM	N125945-0 N123234-0 N87855-0 N31879-1 N55792-0 N86429-0 N99177-0 N3718-0 N47026-0 N67747-0 N107637-0 N54015-0 N65658-0 N117260-0 N112156-0 N23086-0
#665	U260813	11/11/2019 12:18:01 PM	N3718-1 N110341-0 N107637-0 N98178-0 N55792-0 N86429-0 N111088-0 N117260-0 N65552-0 N19805-0 N65658-0 N31879-1 N125945-0 N67747-0 N115772-0 N33037-0 N89651-0 N52856-0 N47257-0 N91047-0 N54015-0 N24150-0 N87855-0 N23086-0 N96337-0 N112156-0 N93674-0 N15053-0 N48364-0 N36306-0 N123234-0 N99177-0 N47026-1
#812	U323740	11/11/2019 12:02:49 PM	N84092-0 N123234-0 N40122-1 N55792-0 N38304-1 N11148-0 N97369-0 N46945-0 N23689-0 N119637-0 N76721-0 N22796-0 N82404-0 N67747-0 N113741-0 N125945-0 N3718-0 N47026-0 N87855-1 N8719-0 N98178-0 N425-0 N116323-0 N117260-1 N99177-0 N86429-0 N31879-1 N65658-0 N13761-0 N2020-0 N17373-0 N65552-0 N107637-0 N31958-0 N54015-0 N47257-0 N23086-0 N30119-0 N50244-1
#879	U655846	11/11/2019 12:17:19 PM	N31879-1 N87855-0 N67747-0 N125945-0 N3718-0 N55792-0 N47026-0 N123234-0
#1154	U14119	11/11/2019 1:47:20 PM	N72801-0 N93674-0 N60389-0 N85452-0 N106586-0 N57903-0 N36905-0 N10057-0 N95770-0 N4872-0 N25597-0 N50244-0 N26887-0 N37106-0 N83421-0 N123362-0 N109137-0 N70822-0 N38304-0 N91892-0 N98178-0 N56410-0 N31879-1 N47110-0 N93643-0 N15847-0 N20648-0 N82072-0 N17575-0 N46945-0 N9250-0 N24150-0 N76489-0 N11682-0 N58417-0 N26404-0 N5841-0 N41134-0 N55944-0 N71607-0 N119637-0 N2319-0 N79081-0 N37376-0 N128940-0 N69915-0 N115676-0 N2020-0 N85063-0 N107088-0 N8719-0 N44136-0 N116064-0 N91047-0 N104437-0
#1192	U481290	11/11/2019 5:10:18 AM	N31879-1 N113609-0 N42197-0 N23920-0 N110341-0 N50360-0 N92260-0 N95962-0 N5568-0 N113703-0 N122543-0 N115958-0 N42694-0 N3664-0 N40500-0 N109606-0 N66169-0 N31323-0 N92438-1 N123643-0 N123172-0 N94839-0 N78111-0 N99486-0 N61323-0 N129918-0 N32154-0 N121774-0 N52807-0 N37861-0 N13229-0 N54441-0 N46945-0 N58102-0 N4858-0 N425-0 N60903-0 N34374-0 N70786-0 N13366-0 N124630-0 N83421-0 N7551-0 N6982-0 N35236-0 N20849-0 N3936-0 N17575-0 N32544-0 N109605-0 N84137-1 N46121-0 N62120-0 N129503-0 N14509-0 N89210-0 N66531-0 N70497-0 N23067-0 N109796-0 N99177-0 N19368-0 N3768-0 N1304-0 N108645-0 N89451-0 N15415-0 N18999-0 N43087-0 N39363-0 N56829-0 N108809-0 N97935-0 N77991-0 N93154-0 N57178-0 N11296-0 N64888-0 N25037-0 N31492-0 N1328-0 N54671-0 N60823-0 N58417-0 N100449-0 N37106-0 N69621-0 N55944-0 N85366-0 N1679-0 N87601-0 N56196-1 N68763-0 N63833-0 N24483-0 N105661-0 N30605-0 N87236-0 N29852-0 N73137-0 N37105-0 N64065-0 N28421-0 N43835-0 N28009-0 N81218-0 N32609-0 N128124-0 N67369-0 N122592-0 N13888-0 N30580-0 N9433-0 N112050-0 N20568-0 N17373-0 N111291-0 N30119-0 N37658-0 N125974-0 N31855-0
#1368	U501140	11/11/2019 12:53:33 PM	N49466-0 N47026-0 N93643-0 N17575-0 N58119-0 N4872-0 N106586-0 N123362-0 N33037-0 N65552-0 N23086-0 N104568-0 N54015-0 N107637-0 N102079-0 N98178-0 N123077-0 N38903-0 N125945-0 N93755-0 N52856-0 N87855-0 N117260-0 N47505-0 N99177-0 N8719-0 N119637-0 N123234-0 N31879-1 N45483-0 N22752-0 N4612-0 N37376-0 N48364-0 N55792-0 N56410-0 N47257-0 N112156-0 N85452-0
#1391	U634759	11/11/2019 12:56:21 PM	N25145-0 N77287-0 N91892-0 N123234-0 N23332-0 N125974-0 N49168-0 N55764-0 N56410-0 N78111-0 N47026-0 N11789-0 N91923-0 N117260-0 N31879-1 N87855-0 N99177-0 N31958-0 N90533-0 N94356-0 N54493-0 N55792-0 N72485-0 N83702-0 N95938-0 N33037-0 N123077-0 N60903-0 N8475-0 N65552-0 N107637-0 N96337-0 N67491-0 N107191-0 N88062-0 N112156-0 N127457-0 N71827-0 N91141-0 N96233-0 N23086-0 N46420-0 N85452-0 N15268-0 N54015-0 N118259-0 N52856-0
#1400	U640989	11/11/2019 1:43:59 PM	N59912-0 N107637-0 N47257-0 N67128-0 N10057-0 N5841-0 N47026-0 N49466-0 N123362-0 N48364-0 N87855-0 N26404-0 N8719-0 N31879-1 N123234-0 N24064-0 N57903-0 N20648-0 N50244-0 N45483-0 N44136-0 N128389-0 N425-0 N49168-0 N14338-0 N24150-0 N56410-0 N98178-0 N56278-0 N94356-0 N23086-0 N15053-1 N85986-0 N9935-0 N30483-0 N4872-0 N66957-0 N85063-0 N66138-0 N13750-0 N116064-0 N11789-0 N23689-0 N36306-0 N1341-0 N42178-0 N58417-0 N64730-0 N23332-0 N52856-0 N67265-0 N8475-0 N55792-0 N54493-0 N60903-0 N33037-0 N11682-0 N117183-0 N77739-0 N30119-0 N76721-0 N63015-0 N65552-0 N2020-0 N75391-0 N127986-0 N117260-0 N4612-0 N68562-0 N70948-0 N70822-0 N86828-0 N46945-0 N28009-0 N77011-0 N90279-0 N85452-0 N80513-0 N40122-0 N123077-0 N111088-0 N17575-0

ids = ['N3718', 'N40122', 'N38304', 'N87855', 'N92438', 'N117260', 'N47026', 'N84137', 'N56196']

for id in ids:
    get_news_info_by_id(id).show(9)



----------------------------------------------------------
+--------+--------------+--------------------+
|category|   subcategory|               title|
+--------+--------------+--------------------+
| finance|finance-career|Hey, Millennials,...|
+--------+--------------+--------------------+

+--------+------------+--------------------+
|category| subcategory|               title|
+--------+------------+--------------------+
|      tv|tv-celebrity|Jordyn Woods debu...|
+--------+------------+--------------------+

+---------+--------------------+-----------------+
| category|         subcategory|            title|
+---------+--------------------+-----------------+
|lifestyle|lifestylepetsanimals|Why Do Cats Meow?|
+---------+--------------------+-----------------+

+--------+-----------+--------------------+
|category|subcategory|               title|
+--------+-----------+--------------------+
|    news|     newsus|NRA turmoil creat...|
+--------+-----------+--------------------+

